# 09 — Exportar as entregas (GeoPackage, Excel e Parquet do Monitor)

**Objetivo:** gravar a base final nos formatos de entrega.

**Entradas:** `df_cnpj_espacial` (08).

**Saídas (`04_entrega/`):** `cnpj_espacial.gpkg`, `cnpj_espacial.xlsx`, `cnpjs_monitor.parquet` e `cnpjs_monitor_meta.json` (copiar os dois últimos para o projeto `monitor_negocios`).

**Como rodar:** de cima para baixo. **Dados internos da Prefeitura:** `04_entrega/` fica fora do Git.

**Seções:** 1. GeoPackage · 2. Planilha Excel · 3. Parquet do Monitor de Negócios · 4. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, datas, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("09_exportar_entregas", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

from src.cascata import ORDEM_CONFIANCA   # ordem das confianças (muito alta ... muito baixa)

# ===== PARÂMETROS (config.yaml) =====
CRS_SAIDA_EPSG = int(cfg.projeto("crs_graus").split(":")[1])      # 4674 = SIRGAS 2000
ARQ_GPKG = cfg.caminho("entrega.gpkg")                             # 04_entrega/cnpj_espacial.gpkg
CAMADA_GPKG = cfg.camada("entrega.gpkg")
ARQ_XLSX = cfg.caminho("entrega.xlsx")                             # 04_entrega/cnpj_espacial.xlsx
PASTA_MONITOR = cfg.caminho("entrega.monitor").parent              # 04_entrega
NOME_PARQUET_MONITOR = cfg.caminho("entrega.monitor").name
NOME_META_MONITOR = cfg.caminho("entrega.monitor_meta").name
ROTULO_BASE_RECEITA = cfg.parametro("monitor.rotulo_base_receita")         # nome da base mostrado no Monitor
COLUNAS_FORA_DO_MONITOR = cfg.parametro("monitor.colunas_fora_do_monitor")  # CNAE secundário e contatos

# ===== ENTRADAS =====
df_cnpj_espacial = fontes.ler_processamento(cfg, "df_cnpj_espacial")   # 08

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. GeoPackage com todos os CNPJs
Grava o `df_cnpj_espacial` em um arquivo GeoPackage (`.gpkg`), com um ponto (longitude = X, latitude = Y) para cada CNPJ que tem coordenada. Os CNPJs sem coordenada entram na mesma camada, sem geometria.
O arquivo é salvo em `04_entrega/` (caminho em `entrega.gpkg` do config.yaml). O log mostra o caminho completo.
Mantém os nomes originais de todas as colunas (inclusive `metodo`, `confianca` e `motivo_sem_coord`) e todos os textos por inteiro. Depois de gravar, reabre o arquivo e confere linhas, geometrias vazias, posição dos pontos e sistema de coordenadas.
As datas (`data_abertura`, `data_encerramento`...) vão como colunas de data. Não altera `df_cnpj_espacial`. Exige a biblioteca `geopandas`. Não exibe linhas dos DataFrames, só logs.

In [ ]:
# Biblioteca para trabalhar com caminhos de pastas e arquivos (já importada antes, repetida aqui por segurança).
from pathlib import Path

log.info("SEÇÃO 1: gerando o GeoPackage...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b30 = time.time()

# ===== CONFIGURAÇÃO (vem do config.yaml: entrega.gpkg e projeto.crs_graus) =====
# Pasta e nome do arquivo (04_entrega/cnpj_espacial.gpkg) e nome da camada dentro dele.
PASTA_SAIDA = ARQ_GPKG.parent
NOME_ARQUIVO = ARQ_GPKG.name
NOME_CAMADA = CAMADA_GPKG
# Sistema de coordenadas dos pontos (4674 = SIRGAS 2000, o mesmo das coordenadas calculadas).
# As coordenadas não são recalculadas: é só a etiqueta do arquivo.
CRS_EPSG = CRS_SAIDA_EPSG
# ===========================================

# ===================================================================
# ETAPA 1: VERIFICAR A BIBLIOTECA
# ===================================================================
log.info("Etapa 1: verificando se a biblioteca geopandas está instalada...")

try:
    import geopandas as gpd
except ImportError:
    # Para o bloco aqui, sem gravar nada, e explica como resolver.
    raise ImportError("Bloco interrompido: a biblioteca geopandas não está instalada (pip install geopandas). Nada foi gravado.")

log.info("Etapa 1 concluída! geopandas encontrada.")

# ===================================================================
# ETAPA 2: CRIAR A GEOMETRIA
# ===================================================================
log.info("\nEtapa 2: criando um ponto para cada CNPJ com coordenada...")

# Trabalha em uma CÓPIA, para o df_cnpj_espacial não ser alterado.
tabela = df_cnpj_espacial.copy()

# Verdadeiro para quem tem latitude E longitude.
com_coord = tabela["lat"].notna() & tabela["lon"].notna()

# Cria os pontos só para quem tem coordenada. ATENÇÃO à ordem: points_from_xy recebe (X, Y) = (longitude, latitude).
pontos = gpd.points_from_xy(tabela.loc[com_coord, "lon"], tabela.loc[com_coord, "lat"])

# Lista com uma posição para cada CNPJ, começando toda vazia (None = sem geometria).
geometrias = np.empty(len(tabela), dtype=object)
# Coloca os pontos nas posições de quem tem coordenada; os demais continuam vazios.
geometrias[com_coord.values] = np.asarray(pontos, dtype=object)

# Monta o GeoDataFrame (tabela com geometria) e define o sistema de coordenadas.
gdf = gpd.GeoDataFrame(
    tabela,
    geometry=gpd.GeoSeries(geometrias, index=tabela.index),
    crs=f"EPSG:{CRS_EPSG}",
)

# O passo (1 a 8) vira número inteiro que aceita vazio (Int64), em vez de texto misturado com vazios.
# Isso é feito só nesta cópia.
gdf["passo"] = pd.to_numeric(gdf["passo"], errors="coerce").astype("Int64")

log.info(f"Etapa 2 concluída! {int(com_coord.sum()):,} pontos criados e "
      f"{int((~com_coord).sum()):,} CNPJs sem geometria.")

# ===================================================================
# ETAPA 3: GRAVAR O ARQUIVO
# ===================================================================
# Caminho completo do arquivo (resolve() mostra o caminho absoluto, sem "atalhos" como ".").
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_SAIDA = (Path(PASTA_SAIDA) / NOME_ARQUIVO).resolve()
log.info(f"\nEtapa 3: gravando {ARQUIVO_SAIDA}...")

# Se já existir um arquivo com esse nome (de uma rodada anterior), ele é apagado para ser recriado do zero.
if ARQUIVO_SAIDA.exists():
    log.info("    Arquivo anterior encontrado: apagando para recriar...")
    ARQUIVO_SAIDA.unlink()

# Grava a camada no GeoPackage. Os nomes das colunas e os textos ficam exatamente como estão.
gdf.to_file(ARQUIVO_SAIDA, layer=NOME_CAMADA, driver="GPKG")

tamanho_mb = ARQUIVO_SAIDA.stat().st_size / 1_000_000
log.info(f"Etapa 3 concluída! Arquivo gravado ({tamanho_mb:,.1f} MB).")

# ===================================================================
# ETAPA 4: REABRIR O ARQUIVO E CONFERIR
# ===================================================================
log.info("\nEtapa 4: reabrindo o arquivo para conferir...")

# Lê de volta a camada que acabamos de gravar.
lido = gpd.read_file(ARQUIVO_SAIDA, layer=NOME_CAMADA)

# Quem tem geometria no arquivo lido (não vazia).
tem_geom = lido.geometry.notna() & ~lido.geometry.is_empty

# Posição dos pontos: X do ponto deve ser a longitude e Y deve ser a latitude (só de quem tem geometria).
dif_x = (lido.loc[tem_geom].geometry.x - lido.loc[tem_geom, "lon"]).abs().max()
dif_y = (lido.loc[tem_geom].geometry.y - lido.loc[tem_geom, "lat"]).abs().max()

# Colunas do df_cnpj_espacial que existem no arquivo lido.
colunas_origem = list(df_cnpj_espacial.columns)
faltam_colunas = [c for c in colunas_origem if c not in lido.columns]

# O CRS lido pode vir como EPSG:4674 ou como um texto equivalente; to_epsg() tenta reconhecer o código.
epsg_lido = lido.crs.to_epsg() if lido.crs is not None else None

log.info("Etapa 4 concluída!")

log.info("\n--- CONFERÊNCIAS DO ARQUIVO REABERTO ---")
log.info(f"Linhas: {len(lido):,} (esperado: {len(df_cnpj_espacial):,}): "
      f"{'confere' if len(lido) == len(df_cnpj_espacial) else 'NÃO confere'}")
log.info(f"Com geometria: {int(tem_geom.sum()):,} (esperado: {int(com_coord.sum()):,}): "
      f"{'confere' if int(tem_geom.sum()) == int(com_coord.sum()) else 'NÃO confere'}")
log.info(f"Sem geometria: {int((~tem_geom).sum()):,} (esperado: {int((~com_coord).sum()):,}): "
      f"{'confere' if int((~tem_geom).sum()) == int((~com_coord).sum()) else 'NÃO confere'}")
log.info(f"Maior diferença entre X do ponto e a coluna lon: {dif_x:.8f} | entre Y e a coluna lat: {dif_y:.8f} "
      f"({'confere' if max(dif_x, dif_y) < 1e-6 else 'NÃO confere'}, tolerância de 0,000001 grau)")
log.info(f"Colunas do df_cnpj_espacial presentes no arquivo: "
      f"{len(colunas_origem) - len(faltam_colunas)} de {len(colunas_origem)}: "
      f"{'confere' if not faltam_colunas else 'FALTAM ' + str(faltam_colunas)}")
log.info(f"CNPJs na mesma ordem do df_cnpj_espacial: "
      f"{'confere' if bool((lido['cnpj'].values == df_cnpj_espacial['cnpj'].values).all()) else 'NÃO confere'}")
log.info(f"Sistema de coordenadas: EPSG:{epsg_lido} (esperado: EPSG:{CRS_EPSG}): "
      f"{'confere' if epsg_lido == CRS_EPSG else 'NÃO confere'}")

log.info("\n--- RESUMO ---")
log.info(f"Arquivo: {ARQUIVO_SAIDA}")
log.info(f"Camada: {NOME_CAMADA} | {len(lido):,} linhas | {lido.shape[1]} colunas (inclui a geometria)")
log.info(f"Tamanho: {tamanho_mb:,.1f} MB")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b30)}.")

## 2. Planilha Excel (xlsx) com todos os CNPJs
Grava o `df_cnpj_espacial` em uma planilha Excel (`cnpj_espacial.xlsx`), com as mesmas colunas e na mesma ordem do GeoPackage, menos a coluna de geometria (a posição fica nas colunas `lat` e `lon`). Entram os CNPJs com e sem coordenada.
As datas (`data_situacao`, `data_abertura`, `data_encerramento`, `data_opcao_simples`...) vão como **datas do Excel** (dd/mm/aaaa). `lat`, `lon`, `espalhamento_m`, `distancia_bases_m`, `nota_fuzzy`, `passo`, `ajuste_limite_m`, `lat_original`, `lon_original`, `passo_original`, `cad_n_inscricoes`, `ano_abertura` e `ano_encerramento` vão como números. O resto vai como texto, sem conversão (códigos com zeros à esquerda, como `situacao_cadastral`, continuam texto). A planilha tem uma segunda aba, `legenda`, explicando as colunas de localização e todos os métodos da cascata.
O arquivo é salvo em `04_entrega/`, junto com o GeoPackage. Depois de gravar, reabre a planilha e confere cabeçalho, linhas e algumas colunas.
Não altera `df_cnpj_espacial`. Exige a biblioteca `xlsxwriter`. Não exibe linhas dos DataFrames, só logs.

In [ ]:
# Biblioteca para trabalhar com caminhos de pastas e arquivos (repetida aqui por segurança).
from pathlib import Path
# Biblioteca para tabelas (DataFrames) e para cálculos com números (repetidas aqui por segurança).
import pandas as pd
import numpy as np
# Biblioteca para medir quanto tempo cada etapa leva.
import time

log.info("SEÇÃO 2: gerando a planilha Excel...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b31 = time.time()

# ===== CONFIGURAÇÃO (ajuste se quiser) =====
# Pasta e nome do arquivo: 04_entrega/cnpj_espacial.xlsx (entrega.xlsx do config.yaml).
PASTA_SAIDA = ARQ_XLSX.parent
NOME_ARQUIVO = ARQ_XLSX.name
# Nome da aba principal.
NOME_ABA = "cnpj_espacial"
# Quantas linhas gravar por vez (só para o log mostrar o andamento).
LINHAS_POR_PARTE = 50_000
# Colunas que vão como NÚMERO na planilha (todas as outras vão como texto).
COLUNAS_NUMERICAS = ["lat", "lon", "espalhamento_m", "nota_fuzzy", "distancia_bases_m", "passo",
                     "ajuste_limite_m", "lat_original", "lon_original", "passo_original", "cad_n_inscricoes",
                     "ano_abertura", "ano_encerramento"]
# Colunas que vão como DATA do Excel (vêm como datas do notebook 08; vazio fica célula vazia).
COLUNAS_DATA = ["data_situacao", "data_inicio_atividade", "data_situacao_especial", "data_opcao_simples",
                "data_exclusao_simples", "data_opcao_mei", "data_exclusao_mei", "data_abertura", "data_encerramento"]
# Como a data aparece no Excel.
FORMATO_DATA_EXCEL = "dd/mm/yyyy"
# Limites do Excel: linhas por aba e caracteres por célula.
LIMITE_LINHAS_EXCEL = 1_048_576
LIMITE_CARACTERES_CELULA = 32_767
# Caracteres de controle invisíveis que o Excel não aceita (exemplo: \x00 a \x08, \x0b, \x0c e \x0e a \x1f).
CARACTERES_INVALIDOS = r"[\x00-\x08\x0b\x0c\x0e-\x1f]"
# ===========================================

# ===================================================================
# ETAPA 1: VERIFICAR AS BIBLIOTECAS E OS DADOS
# ===================================================================
log.info("Etapa 1: verificando as bibliotecas e os dados...")

# xlsxwriter grava a planilha. Se não estiver instalada, o bloco para aqui, sem gravar nada.
try:
    import xlsxwriter
except ImportError:
    raise ImportError("Bloco interrompido: a biblioteca xlsxwriter não está instalada (pip install xlsxwriter). Nada foi gravado.")

# openpyxl serve só para reabrir a planilha na conferência final. Se faltar, a conferência é pulada.
try:
    import openpyxl
    TEM_OPENPYXL = True
except ImportError:
    TEM_OPENPYXL = False
    log.warning("    AVISO: a biblioteca openpyxl não está instalada (pip install openpyxl). A conferência final será pulada.")

# As colunas numéricas combinadas precisam existir no df_cnpj_espacial.
faltam = [c for c in COLUNAS_NUMERICAS + COLUNAS_DATA if c not in df_cnpj_espacial.columns]
if faltam:
    raise ValueError(f"Bloco interrompido: faltam colunas no df_cnpj_espacial: {faltam}. Nada foi gravado.")

# Total de linhas e de colunas; a aba do Excel só comporta até LIMITE_LINHAS_EXCEL linhas (contando o cabeçalho).
total_linhas = len(df_cnpj_espacial)
colunas = list(df_cnpj_espacial.columns)
if total_linhas + 1 > LIMITE_LINHAS_EXCEL:
    raise ValueError(f"Bloco interrompido: {total_linhas:,} linhas não cabem em uma aba do Excel. Nada foi gravado.")

# Colunas de texto: todas as que não são numéricas nem de data.
colunas_texto = [c for c in colunas if c not in COLUNAS_NUMERICAS and c not in COLUNAS_DATA]

log.info(f"Etapa 1 concluída! {total_linhas:,} linhas e {len(colunas)} colunas "
      f"({len(colunas_texto)} de texto, {len(COLUNAS_NUMERICAS)} numéricas e {len(COLUNAS_DATA)} de data).")

# ===================================================================
# ETAPA 2: PREPARAR O ARQUIVO
# ===================================================================
# Caminho completo do arquivo (resolve() mostra o caminho absoluto, sem "atalhos" como ".").
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_SAIDA = (Path(PASTA_SAIDA) / NOME_ARQUIVO).resolve()
log.info(f"\nEtapa 2: preparando o arquivo {ARQUIVO_SAIDA}...")

# Se já existir um arquivo com esse nome (de uma rodada anterior), ele é apagado para ser recriado do zero.
if ARQUIVO_SAIDA.exists():
    log.info("    Arquivo anterior encontrado: apagando para recriar...")
    try:
        ARQUIVO_SAIDA.unlink()
    except PermissionError:
        # O Windows não deixa apagar um arquivo que está aberto no Excel.
        raise PermissionError(f"Bloco interrompido: não foi possível apagar {ARQUIVO_SAIDA}. "
                              "Feche o arquivo no Excel e rode de novo.")

log.info("Etapa 2 concluída!")

# ===================================================================
# ETAPA 3: GRAVAR A ABA PRINCIPAL
# ===================================================================
log.info("\nEtapa 3: gravando a aba principal, em partes...")

# Opções da planilha:
# constant_memory: grava linha por linha, sem guardar tudo na memória (importante para tabelas grandes);
# strings_to_formulas e strings_to_urls desligados: um texto que comece com "=" ou pareça um link
#   continua sendo só texto, e não vira fórmula nem link;
# nan_inf_to_errors: proteção extra para valores numéricos inválidos;
# default_date_format: como as datas aparecem (dia/mês/ano).
opcoes = {
    "constant_memory": True,
    "strings_to_formulas": False,
    "strings_to_urls": False,
    "nan_inf_to_errors": True,
    "default_date_format": FORMATO_DATA_EXCEL,
}
workbook = xlsxwriter.Workbook(str(ARQUIVO_SAIDA), opcoes)

# Formato do cabeçalho: texto em negrito.
formato_negrito = workbook.add_format({"bold": True})

# Cria a aba principal e escreve a linha de títulos na primeira linha (linha 0).
aba = workbook.add_worksheet(NOME_ABA)
aba.write_row(0, 0, colunas, formato_negrito)
# Colunas de data mais largas, para a data não aparecer como "########".
for coluna_data in COLUNAS_DATA:
    posicao = colunas.index(coluna_data)
    aba.set_column(posicao, posicao, 12)

# Contadores de conferência: células com caracteres inválidos (por coluna) e células acima do limite.
invalidos_por_coluna = {}
celulas_longas_por_coluna = {}

# Passa pelo df_cnpj_espacial em partes de LINHAS_POR_PARTE linhas.
for inicio in range(0, total_linhas, LINHAS_POR_PARTE):
    fim = min(inicio + LINHAS_POR_PARTE, total_linhas)

    # Copia só esta parte (o df_cnpj_espacial original não é alterado).
    parte = df_cnpj_espacial.iloc[inicio:fim].copy()

    # Colunas de texto: conta e remove os caracteres inválidos, e conta as células longas demais.
    for coluna in colunas_texto:
        serie = parte[coluna]
        # Colunas numéricas (por exemplo, uma coluna toda vazia) não têm o que limpar.
        if pd.api.types.is_numeric_dtype(serie):
            continue
        # Verdadeiro para as células com algum caractere inválido.
        com_invalido = serie.str.contains(CARACTERES_INVALIDOS, regex=True, na=False)
        if com_invalido.any():
            invalidos_por_coluna[coluna] = invalidos_por_coluna.get(coluna, 0) + int(com_invalido.sum())
            # Tira só os caracteres inválidos; o resto do texto fica como está.
            parte[coluna] = serie.str.replace(CARACTERES_INVALIDOS, "", regex=True)
        # Conta as células com mais caracteres que o limite do Excel (só avisa, não corta nada).
        longas = int((serie.str.len() > LIMITE_CARACTERES_CELULA).sum())
        if longas > 0:
            celulas_longas_por_coluna[coluna] = celulas_longas_por_coluna.get(coluna, 0) + longas

    # Colunas numéricas: força o tipo número (o que não for número vira vazio).
    for coluna in COLUNAS_NUMERICAS:
        parte[coluna] = pd.to_numeric(parte[coluna], errors="coerce").astype("float64")

    # Colunas de data: garante o tipo data (sem data fica vazio).
    for coluna in COLUNAS_DATA:
        parte[coluna] = datas.para_data(parte[coluna])

    # Troca os valores vazios (NaN) por None, que o xlsxwriter grava como célula vazia.
    parte = parte.astype(object)
    parte = parte.where(parte.notna(), None)

    # Transforma a parte em uma lista de linhas (cada linha é uma lista de valores).
    linhas = parte.values.tolist()

    # Grava linha por linha. A linha 0 é o cabeçalho, então os dados começam na linha 1.
    for deslocamento, linha in enumerate(linhas):
        aba.write_row(1 + inicio + deslocamento, 0, linha)

    # Mostra o andamento (flush=True faz a mensagem aparecer na hora).
    log.info(f"    {fim:,} de {total_linhas:,} linhas gravadas...")

# Congela a primeira linha (o cabeçalho fica sempre visível ao rolar) e liga o filtro automático nas colunas.
aba.freeze_panes(1, 0)
aba.autofilter(0, 0, total_linhas, len(colunas) - 1)

log.info("Etapa 3 concluída! Aba principal gravada (ainda falta fechar o arquivo).")

# ===================================================================
# ETAPA 4: GRAVAR A ABA DE LEGENDA
# ===================================================================
log.info("\nEtapa 4: gravando a aba de legenda...")

# Cada item é (campo, valor, significado). Os limites citados vêm dos notebooks 03 e 04 e foram escolhidos
# no script, ainda não validados.
legenda = [
    ("cnpj", "", "CNPJ de 14 dígitos (básico + ordem + dígito verificador), sem máscara."),
    ("lat / lon", "", "Latitude e longitude do ponto, no sistema de coordenadas indicado no GeoPackage. "
                      "Vêm da base da Prefeitura (SIRF + lotes) ou do CNEFE (IBGE)."),
    ("base", "Prefeitura", "O ponto veio da base da Prefeitura: endereços do SIRF com o ponto do lote georreferenciado."),
    ("base", "CNEFE", "O ponto veio do CNEFE (IBGE)."),
    ("base", "Prefeitura + CNEFE", "As duas bases resolveram o mesmo endereço e apontaram praticamente o mesmo lugar."),
    ("base", "OpenStreetMap (Nominatim)", "O ponto veio do geocoding no OpenStreetMap (passo 13). "
                                         "Dados © colaboradores do OpenStreetMap, licença ODbL."),
    ("base", "CNPJs já localizados", "Centro do bairro ou do município, calculado com os CNPJs já localizados (passos 14 e 15)."),
    ("passo", "1", "CEP + número encontrados na base, com o nome da via igual."),
    ("passo", "2", "CEP + número encontrados na base, com o nome da via parecido (erro de digitação na Receita)."),
    ("passo", "3", "Via + número encontrados na base, em outro CEP: o CEP da Receita foi considerado errado."),
    ("passo", "4", "Interpolação entre números vizinhos, na mesma via e no mesmo CEP."),
    ("passo", "5", "Via de nome parecido com o mesmo número, em qualquer CEP (nome da via e CEP da Receita corrigidos)."),
    ("passo", "6", "Nome da via corrigido dentro do mesmo CEP e, depois, interpolação entre os números vizinhos."),
    ("passo", "7", "CEP corrigido (a via tem um único CEP na base) e, depois, interpolação entre os números vizinhos."),
    ("passo", "8", "Ponto central da via naquele CEP (média de todos os endereços da base com aquele CEP e aquela via)."),
    ("passo", "9", "Núcleo do nome da via (sem tipo, títulos e preposições) igual ao da base: CEP + número, interpolação, "
                   "número exato com CEP corrigido, interpolação com CEP corrigido ou na via inteira, ou centro da via."),
    ("passo", "10", "Como o passo 9, depois de corrigir o núcleo do nome (iniciais, como JOAO B NOGUEIRA, ou grafia parecida). "
                    "Confiança no máximo baixa."),
    ("passo", "11", "A via foi achada na base de CEPs dos Correios: o ponto é o centro do CEP dela (só CEP compacto)."),
    ("passo", "12", "Centro do CEP da Receita (só CEP confiável e compacto, com até 500 m de espalhamento)."),
    ("passo", "13", "Geocoding no OpenStreetMap (Nominatim), aceito só na cidade e com o nome da rua parecido com o da Receita."),
    ("passo", "14", "Centro do bairro: mediana dos CNPJs do mesmo bairro já localizados com confiança média ou melhor."),
    ("passo", "15", "Centro do município: o CNPJ não tinha nenhuma informação de localização utilizável."),
    ("passo", "(vazio)", "Sem coordenada. O motivo está em motivo_sem_coord."),
    ("confianca", "muito alta", "As duas bases resolveram o endereço e os pontos ficaram próximos um do outro."),
    ("confianca", "alta", "Uma base, com CEP + número + via batendo e os pontos usados bem concentrados."),
    ("confianca", "média", "Número exato, mas com os pontos mais espalhados, ou com o CEP corrigido, "
                           "ou interpolação entre vizinhos próximos de mesma paridade."),
    ("confianca", "baixa", "Interpolação em condições piores, nome da via ou CEP corrigidos sem confirmação, "
                           "centro de uma via curta, centro de um CEP de até 300 m ou geocoding com o número encontrado."),
    ("confianca", "muito baixa", "Centro de uma via longa, de um CEP de 301 a 500 m, do bairro ou do município, ou "
                                 "geocoding só pela rua: serve para saber a região, não o lote. Cuidado em análises de densidade."),
    ("metodo", "", "Texto completo: qual base resolveu, qual passo e em que condição. "
                   "Exemplo: 'Prefeitura: CEP + número, via igual — pontos concentrados'."),
    ("espalhamento_m", "", "Distância, em metros, entre os pontos usados para calcular a coordenada "
                           "(nos passos de interpolação, a distância entre os dois vizinhos; "
                           "no passo 8, a extensão da via naquele CEP)."),
    ("distancia_bases_m", "", "Só quando as duas bases resolveram o mesmo endereço: distância, em metros, "
                              "entre o ponto da Prefeitura e o do CNEFE."),
    ("cep_usado", "", "CEP de fato usado para achar o ponto: o da Receita, quando ele conferiu, "
                      "ou o CEP da base, quando o da Receita foi corrigido."),
    ("via_candidata / nota_fuzzy", "", "Quando o nome da via foi corrigido: o nome escolhido na base "
                                       "e a nota de semelhança (0 a 100)."),
    ("motivo_sem_coord", "", "Por que o CNPJ ficou sem coordenada (cadastro incompleto, ou endereço "
                             "que não existe em nenhuma das duas bases)."),
    ("<campo>_nome", "", "Descrição do código que está na coluna ao lado, pelos arquivos da Receita: natureza_juridica_nome, "
                         "motivo_situacao_nome, pais_nome, cnae_principal_nome e cnae_secundaria_nome (lista 'código – descrição' separada por ' | '), "
                         "ou pelo layout da Receita: situacao_nome, porte_nome, matriz_filial_nome, opcao_simples_nome e opcao_mei_nome "
                         "(Sim, Não ou Outros; vazio quando o CNPJ não tem registro no arquivo do Simples). "
                         "Código sem descrição aparece como '(código fora da tabela)' ou '(código fora do layout)'."),
    ("data_abertura / ano_abertura / ano_mes_abertura", "", "Data de início da atividade (a mesma data_inicio_atividade), o ano e o mês no formato AAAA-MM."),
    ("data_encerramento / ano_encerramento / ano_mes_encerramento", "", "Só para estabelecimentos com situação Baixada: a data da baixa "
                                                                      "(a data_situacao), o ano e o mês no formato AAAA-MM. Vazias para os demais."),
    ("cnae_subclasse / cnae_denominacao", "", "CNAE principal do estabelecimento, pela tabela de subclasses do IBGE (cnae_principal_nome vem da Receita)."),
    ("cnae_secao / cnae_divisao", "", "Níveis acima do CNAE principal, com o código e a denominação de cada um."),
    ("cnae_origem", "", "De onde veio a classificação do CNAE: 'tabela IBGE'; 'não informado' (código 8888888 da Receita, "
                        "com seção e divisão ND); 'divisão pelos 2 primeiros dígitos' (subclasse fora da tabela); "
                        "ou 'fora da tabela do IBGE, sem classificação'."),
    ("ajuste_limite / ajuste_limite_m", "", "Pontos que estavam fora do limite municipal (notebook 07 (seção 4)): todos vão para o ponto mais próximo dentro do limite, "
                                            "com método e confiança originais mantidos. Os que estavam a mais de 1.000 m do limite têm 'conferir manualmente' no texto. "
                                            "ajuste_limite_m é a distância, em metros, do ponto original até o limite."),
    ("lat_original / lon_original / passo_original / metodo_original / confianca_original / base_original", "",
     "Como o ponto estava antes do ajuste ao limite municipal (só nos CNPJs ajustados)."),
    ("cep_receita / cep_nao_confiavel", "", "CEP informado pela Receita e se ele é um CEP não confiável ('genérico', como o 12227788): sim ou não."),
    ("cep_corrigido / origem_cep_corrigido", "", "Só para CEP não confiável: o CEP certo achado pelo nome da via e a regra usada "
                                                  "(via com um CEP só nos Correios; bairro desempata; endereço de referência mais próximo na mesma via). "
                                                  "Vazio quando nenhuma regra serviu."),
    ("cad_insc_mun / cad_estabelecido / cad_tipo / cad_reg_tributario", "", "Do cadastro municipal, pelo CNPJ. Valores como estão na planilha. "
                                                                           "Vazio quando o CNPJ não está no cadastro."),
    ("cad_cnae_principal / cad_descricao_cnae / cad_cnae_diverge", "", "CNAE principal do cadastro municipal (só dígitos) e sua descrição. "
                                                                       "cad_cnae_diverge diz se ele é diferente do CNAE da Receita (sim ou não). Os dois CNAEs são mantidos."),
    ("cad_n_inscricoes", "", "Quantas inscrições municipais o CNPJ tem no cadastro. Os dados cad_ são os da primeira linha da planilha."),
    ("demais colunas", "", "Vêm da Receita Federal como estavam: capital_social com vírgula decimal (texto) e códigos "
                           "(como porte e natureza jurídica) como texto. As datas foram convertidas em datas de verdade ('00000000' vira vazio)."),
]

# Cria a aba da legenda e escreve o cabeçalho (em negrito) e as linhas.
aba_legenda = workbook.add_worksheet("legenda")
aba_legenda.write_row(0, 0, ["campo", "valor", "significado"], formato_negrito)
for numero_linha, item in enumerate(legenda, start=1):
    aba_legenda.write_row(numero_linha, 0, list(item))

log.info(f"Etapa 4 concluída! {len(legenda):,} linhas de legenda.")

# ===================================================================
# ETAPA 5: FECHAR O ARQUIVO
# ===================================================================
log.info("\nEtapa 5: fechando o arquivo (é nesta hora que ele é montado, pode demorar)...")

try:
    # close() finaliza a planilha e grava o arquivo em disco.
    workbook.close()
except (PermissionError, xlsxwriter.exceptions.FileCreateError):
    raise PermissionError(f"Bloco interrompido: não foi possível gravar {ARQUIVO_SAIDA}. "
                          "Verifique se o arquivo não está aberto no Excel e se há permissão de escrita na pasta.")

tamanho_mb = ARQUIVO_SAIDA.stat().st_size / 1_000_000
log.info(f"Etapa 5 concluída! Arquivo gravado ({tamanho_mb:,.1f} MB).")

# ===================================================================
# ETAPA 6: REABRIR O ARQUIVO E CONFERIR
# ===================================================================
if TEM_OPENPYXL:
    log.info("\nEtapa 6: reabrindo a planilha para conferir (a leitura pode demorar)...")

    # Lê só o cabeçalho e os nomes das abas, de forma rápida (read_only não carrega a planilha toda).
    livro = openpyxl.load_workbook(ARQUIVO_SAIDA, read_only=True)
    abas_lidas = livro.sheetnames
    cabecalho_lido = list(next(livro[NOME_ABA].iter_rows(min_row=1, max_row=1, values_only=True)))
    livro.close()

    # Lê só 3 colunas inteiras (CNPJ, passo e latitude). O CNPJ é lido como texto, para manter os zeros à esquerda.
    log.info("    Lendo as colunas cnpj, passo e lat...")
    lido = pd.read_excel(ARQUIVO_SAIDA, sheet_name=NOME_ABA, usecols=["cnpj", "passo", "lat"], dtype={"cnpj": str})
    log.info("    Leitura concluída!")

    # Valores de origem, com os mesmos tipos usados na gravação.
    passo_origem = pd.to_numeric(df_cnpj_espacial["passo"], errors="coerce").astype("float64").values
    passo_lido = pd.to_numeric(lido["passo"], errors="coerce").astype("float64").values
    lat_origem = pd.to_numeric(df_cnpj_espacial["lat"], errors="coerce").astype("float64").values
    lat_lido = pd.to_numeric(lido["lat"], errors="coerce").astype("float64").values

    log.info("Etapa 6 concluída!")

    log.info("\n--- CONFERÊNCIAS DA PLANILHA REABERTA ---")
    log.info(f"Abas: {abas_lidas}: {'confere' if abas_lidas == [NOME_ABA, 'legenda'] else 'NÃO confere'}")
    log.info(f"Cabeçalho igual às colunas do df_cnpj_espacial ({len(colunas)} colunas): "
          f"{'confere' if cabecalho_lido == colunas else 'NÃO confere'}")
    log.info(f"Linhas de dados: {len(lido):,} (esperado: {total_linhas:,}): "
          f"{'confere' if len(lido) == total_linhas else 'NÃO confere'}")
    # Só compara os valores se o número de linhas bate (senão a comparação não faz sentido).
    if len(lido) == total_linhas:
        log.info(f"CNPJs iguais e na mesma ordem: "
              f"{'confere' if bool((lido['cnpj'].values == df_cnpj_espacial['cnpj'].values).all()) else 'NÃO confere'}")
        log.info(f"Coluna passo igual: {'confere' if np.array_equal(passo_origem, passo_lido, equal_nan=True) else 'NÃO confere'}")
        log.info(f"Coluna lat igual (tolerância de 0,000000001 grau): "
              f"{'confere' if np.allclose(lat_origem, lat_lido, rtol=0, atol=1e-9, equal_nan=True) else 'NÃO confere'}")
else:
    log.info("\nEtapa 6: pulada (openpyxl não instalada).")

# ===================================================================
# RESUMO
# ===================================================================
log.info("\n--- RESUMO ---")
log.info(f"Arquivo: {ARQUIVO_SAIDA}")
log.info(f"Aba principal: {NOME_ABA} | {total_linhas:,} linhas | {len(colunas)} colunas | aba extra: legenda")
log.info(f"Tamanho: {tamanho_mb:,.1f} MB")

log.info("\nCaracteres inválidos para o Excel (removidos só da planilha; os DataFrames e o GeoPackage não mudam):")
if invalidos_por_coluna:
    for coluna, qtd in invalidos_por_coluna.items():
        log.info(f"  {coluna}: {qtd:,} células")
    log.info(f"  Total: {sum(invalidos_por_coluna.values()):,} células")
else:
    log.info("  Nenhuma célula com caracteres inválidos.")

log.info("\nCélulas acima do limite de caracteres do Excel (apenas aviso, nada foi cortado):")
if celulas_longas_por_coluna:
    for coluna, qtd in celulas_longas_por_coluna.items():
        log.info(f"  {coluna}: {qtd:,} células")
else:
    log.info("  Nenhuma célula acima do limite.")

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b31)}.")

## 3. Parquet para o Monitor de Negócios
Gera o arquivo que o **Monitor de Negócios** (Streamlit) vai ler: `cnpjs_monitor.parquet`, em `04_entrega/`, mais um `cnpjs_monitor_meta.json` com a data da base, a data da geração e as colunas.
Do `df_cnpj_espacial`, entram os dados do estabelecimento, da empresa, do Simples/MEI, do CNAE principal, da localização (inclusive confiança, ajuste ao limite e CEP corrigido) e do cadastro municipal. Saem os CNAEs secundários, e-mail, telefones, fax, país, cidade no exterior e situação especial (lista em `parametros.monitor.colunas_fora_do_monitor`, no config.yaml).
As datas já vêm como data de verdade do notebook 08 (`00000000` virou vazio, sem inventar data), o capital social vira número, `passo` vira inteiro e os campos repetitivos (situação, confiança, seção do CNAE, nomes dos códigos...) viram categorias, para o arquivo ficar pequeno e rápido. Os nomes dos códigos (`situacao_nome`, `porte_nome`, `matriz_filial_nome`, `natureza_juridica_nome`, `motivo_situacao_nome`, `cnae_principal_nome`, `opcao_simples_nome` e `opcao_mei_nome`) e as colunas de abertura e encerramento (`data_abertura`, `ano_abertura`, `data_encerramento`...) também entram; `cnae_secundaria_nome` e `pais_nome` ficam de fora, como os códigos deles. Acrescenta `confianca_nivel` (5 = muito alta, 1 = muito baixa).
Depois de gravar, reabre o arquivo e confere linhas, colunas, CNPJs e coordenadas. Exige `pyarrow`. Não altera `df_cnpj_espacial`. Só logs.

In [ ]:
log.info("SEÇÃO 3: gerando o Parquet para o Monitor de Negócios...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_monitor = time.time()

# ===================================================================
# ETAPA 1: VERIFICAR AS BIBLIOTECAS E OS DADOS
# ===================================================================
log.info("Etapa 1: verificando as bibliotecas e os dados...")
# pyarrow grava e lê o Parquet. Se não estiver instalada, o bloco para aqui, sem gravar nada.
if importlib.util.find_spec("pyarrow") is None:
    raise ImportError("Bloco interrompido: falta a biblioteca pyarrow (py -m pip install --user pyarrow). Nada foi gravado.")
if "df_cnpj_espacial" not in globals():
    raise NameError("Bloco interrompido: o df_cnpj_espacial não existe. Rode o notebook 08 (seção 1) antes.")
# Atalho curto para a tabela (nada será alterado nela: tudo é feito numa cópia).
origem = df_cnpj_espacial
# Colunas obrigatórias para o Monitor funcionar.
obrigatorias = ["cnpj", "lat", "lon", "passo", "confianca", "situacao_cadastral", "situacao_nome", "porte_nome",
                "matriz_filial_nome", "data_situacao", "data_inicio_atividade", "data_abertura", "data_encerramento",
                "cnae_secao", "cnae_origem"]
faltam = [c for c in obrigatorias if c not in origem.columns]
if faltam:
    raise ValueError(f"Bloco interrompido: faltam colunas no df_cnpj_espacial: {faltam}. Nada foi gravado.")
log.info(f"Etapa 1 concluída! {len(origem):,} linhas e {origem.shape[1]} colunas de origem.")

# ===================================================================
# ETAPA 2: ESCOLHER AS COLUNAS
# ===================================================================
log.info("\nEtapa 2: escolhendo as colunas...")
# Cópia só das colunas que vão para o Monitor (as da lista COLUNAS_FORA_DO_MONITOR ficam de fora).
fora_da_lista = [c for c in COLUNAS_FORA_DO_MONITOR if c in origem.columns]
monitor = origem.drop(columns=fora_da_lista).copy()
log.info(f"Etapa 2 concluída! {len(fora_da_lista)} colunas ficam de fora; {monitor.shape[1]} seguem.")

# ===================================================================
# ETAPA 3: CONVERTER OS TIPOS
# ===================================================================
log.info("\nEtapa 3: convertendo datas, números e categorias...")


# Datas: o notebook 08 já entrega datas de verdade; a função só converte o que ainda estiver como texto AAAAMMDD
# (e deixa passar o que já é data), então o Parquet continua certo mesmo com uma base mais antiga.
for coluna in ["data_situacao", "data_inicio_atividade", "data_opcao_simples", "data_exclusao_simples",
               "data_opcao_mei", "data_exclusao_mei", "data_abertura", "data_encerramento"]:
    if coluna in monitor.columns:
        monitor[coluna] = datas.para_data(monitor[coluna])
# Marca quem tinha a data da situação zerada na Receita (para o Monitor saber que não é "sem mudança").
monitor["data_situacao_ausente"] = monitor["data_situacao"].isna()

# Capital social: a Receita grava com vírgula decimal (ex.: "5000,00"). Vira número.
if "capital_social" in monitor.columns:
    capital = monitor["capital_social"].fillna("").astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    monitor["capital_social"] = pd.to_numeric(capital, errors="coerce")

# Os nomes dos códigos (situacao_nome, porte_nome, matriz_filial_nome...) já vêm prontos do notebook 08
# (legendas em parametros.legendas_receita do config.yaml), com a mesma escrita de antes.

# Confiança: categoria com a ordem certa e um número (5 = muito alta ... 1 = muito baixa) para ordenar e filtrar.
monitor["confianca"] = pd.Categorical(monitor["confianca"], categories=ORDEM_CONFIANCA, ordered=True)
monitor["confianca_nivel"] = (len(ORDEM_CONFIANCA) - monitor["confianca"].cat.codes).where(monitor["confianca"].notna()).astype("Int8")

# Passo: número inteiro (no notebook ele é decimal por causa dos vazios).
monitor["passo"] = pd.to_numeric(monitor["passo"], errors="coerce").astype("Int8")
if "passo_original" in monitor.columns:
    monitor["passo_original"] = pd.to_numeric(monitor["passo_original"], errors="coerce").astype("Int8")
# Coordenadas e medidas: número decimal.
for coluna in ["lat", "lon", "espalhamento_m", "distancia_bases_m", "nota_fuzzy", "ajuste_limite_m",
               "lat_original", "lon_original"]:
    if coluna in monitor.columns:
        monitor[coluna] = pd.to_numeric(monitor[coluna], errors="coerce").astype("float64")
if "cad_n_inscricoes" in monitor.columns:
    monitor["cad_n_inscricoes"] = pd.to_numeric(monitor["cad_n_inscricoes"], errors="coerce").astype("Int16")
# Ano de abertura e de encerramento: inteiro pequeno que aceita vazio.
for coluna in ["ano_abertura", "ano_encerramento"]:
    if coluna in monitor.columns:
        monitor[coluna] = pd.to_numeric(monitor[coluna], errors="coerce").astype("Int16")

# Campos com poucos valores repetidos muitas vezes viram categoria (arquivo menor e leitura mais rápida).
colunas_categoria = ["situacao_cadastral", "situacao_nome", "motivo_situacao", "matriz_filial", "matriz_filial_nome",
                     "uf", "porte_empresa", "porte_nome", "natureza_juridica", "opcao_simples", "opcao_mei",
                     "base", "metodo", "cnae_origem", "cnae_secao", "cnae_secao_denominacao", "cnae_divisao",
                     "cnae_divisao_denominacao", "ajuste_limite", "metodo_original", "confianca_original",
                     "base_original", "cep_nao_confiavel", "origem_cep_corrigido", "cad_tipo", "cad_estabelecido",
                     "cad_reg_tributario", "cad_cnae_diverge", "natureza_juridica_nome", "motivo_situacao_nome",
                     "cnae_principal_nome", "opcao_simples_nome", "opcao_mei_nome"]
for coluna in colunas_categoria:
    if coluna in monitor.columns:
        monitor[coluna] = monitor[coluna].astype("category")
log.info("Etapa 3 concluída!")

# ===================================================================
# ETAPA 4: GRAVAR O PARQUET E O ARQUIVO DE INFORMAÇÕES
# ===================================================================
log.info("\nEtapa 4: gravando o Parquet e o arquivo de informações...")
# Cria a pasta, se ainda não existir.
Path(PASTA_MONITOR).mkdir(parents=True, exist_ok=True)
arquivo_parquet = (Path(PASTA_MONITOR) / NOME_PARQUET_MONITOR).resolve()
arquivo_meta = (Path(PASTA_MONITOR) / NOME_META_MONITOR).resolve()
# Grava num arquivo temporário e só depois troca pelo definitivo (nunca fica um arquivo pela metade).
temporario = arquivo_parquet.with_name(arquivo_parquet.name + ".tmp")
try:
    monitor.to_parquet(temporario, engine="pyarrow", compression="snappy", index=False)
except Exception as erro:
    if temporario.exists():
        temporario.unlink()
    raise RuntimeError(f"Bloco interrompido: falha ao gravar o Parquet ({type(erro).__name__}): {str(erro)[:200]}.") from None
try:
    os.replace(temporario, arquivo_parquet)
except PermissionError:
    raise PermissionError(f"Bloco interrompido: não foi possível substituir {arquivo_parquet}. "
                          "Feche o programa que está usando o arquivo (o Monitor, por exemplo) e rode de novo.")

# Data da base: a maior data encontrada nos campos de data da Receita (o Monitor usa como limite dos filtros).
datas_validas = pd.concat([monitor["data_situacao"], monitor["data_inicio_atividade"]]).dropna()
data_base = datas_validas.max().date().isoformat()
informacoes = {
    "rotulo_base": ROTULO_BASE_RECEITA,
    "data_base": data_base,
    "gerado_em": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "linhas": int(len(monitor)),
    "colunas": {c: str(t) for c, t in monitor.dtypes.items()},
    "ordem_confianca": ORDEM_CONFIANCA,
    "sistema_de_coordenadas_epsg": CRS_SAIDA_EPSG,
}
arquivo_meta.write_text(json.dumps(informacoes, ensure_ascii=False, indent=2), encoding="utf-8")
log.info(f"Etapa 4 concluída! Parquet: {arquivo_parquet.stat().st_size / 1e6:.1f} MB | informações: {arquivo_meta.name}.")

# ===================================================================
# ETAPA 5: REABRIR E CONFERIR
# ===================================================================
log.info("\nEtapa 5: reabrindo o Parquet para conferir...")
lido = pd.read_parquet(arquivo_parquet, engine="pyarrow")
log.info("Etapa 5 concluída!")
log.info("\n--- CONFERÊNCIAS DO ARQUIVO REABERTO ---")
log.info(f"Linhas: {len(lido):,} (esperado: {len(monitor):,}): {'confere' if len(lido) == len(monitor) else 'NÃO confere'}")
log.info(f"Colunas na mesma ordem: {'confere' if list(lido.columns) == list(monitor.columns) else 'NÃO confere'}")
log.info(f"CNPJs iguais e na mesma ordem: {'confere' if lido['cnpj'].equals(monitor['cnpj']) else 'NÃO confere'}")
log.info(f"CNPJs todos diferentes: {'confere' if lido['cnpj'].is_unique else 'NÃO confere'}")
log.info(f"Latitude e longitude iguais e sem vazios: "
      f"{'confere' if (lido['lat'].equals(monitor['lat']) and lido['lon'].equals(monitor['lon']) and lido[['lat', 'lon']].notna().all().all()) else 'NÃO confere'}")
vazios_iguais = (lido.isna().sum() == monitor.isna().sum()).all()
log.info(f"Quantidade de vazios igual em todas as colunas: {'confere' if vazios_iguais else 'NÃO confere'}")

log.info("\n--- RESUMO ---")
log.info(f"Arquivo: {arquivo_parquet}")
log.info(f"Linhas: {len(lido):,} | colunas: {lido.shape[1]} | data da base: {data_base}")
log.info("Situação cadastral:")
for nome_sit, qtd in lido["situacao_nome"].value_counts().items():
    log.info(f"  {nome_sit}: {qtd:,}")
log.info("Confiança da localização:")
contagem_conf = lido["confianca"].value_counts()
for nome in ORDEM_CONFIANCA:
    log.info(f"  {nome}: {int(contagem_conf.get(nome, 0)):,}")
log.info(f"Datas: data_inicio_atividade vazias: {int(lido['data_inicio_atividade'].isna().sum()):,} | "
      f"data_situacao vazias: {int(lido['data_situacao'].isna().sum()):,}")
log.info(f"Memória do DataFrame lido: {lido.memory_usage(deep=True).sum() / 1e6:.0f} MB")
log.info("Lembrete: a pasta do Parquet tem dados internos da Prefeitura e precisa estar no .gitignore (linha \"04_entrega/\").")

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_monitor)}.")

## 4. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 09_exportar_entregas...")
log.info("Lembrete: copie cnpjs_monitor.parquet e cnpjs_monitor_meta.json para o projeto monitor_negocios.")
log.info("Arquivos gerados:")
for chave in ['gpkg', 'xlsx', 'monitor', 'monitor_meta']:
    caminho = cfg.caminho("entrega." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")